In [6]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display


def calcular_probabilidades_glauber(J: float, H: float, kBT: float) -> np.ndarray:
    """Precalcula p_j para las 10 clases (indexadas de 0 a 9)."""
    p = np.zeros(10, dtype=np.float64)
    vecinos_arriba = [4, 3, 2, 1, 0]

    for idx, u in enumerate(vecinos_arriba):
        suma_vecinos = 2 * u - 4
        dE_up = 2.0 * (J * suma_vecinos + H)
        p[idx] = 1.0 / (1.0 + math.exp(dE_up / kBT))

        dE_down = -2.0 * (J * suma_vecinos + H)
        p[idx + 5] = 1.0 / (1.0 + math.exp(dE_down / kBT))

    return p


def construir_vecinos(L: int) -> np.ndarray:
    """Precalcula los 4 vecinos de cada sitio i en una red LxL periódica."""
    N = L * L
    vecinos = np.empty((N, 4), dtype=np.int32)
    for r in range(L):
        for c in range(L):
            i = r * L + c
            vecinos[i, 0] = ((r - 1) % L) * L + c
            vecinos[i, 1] = ((r + 1) % L) * L + c
            vecinos[i, 2] = r * L + ((c - 1) % L)
            vecinos[i, 3] = r * L + ((c + 1) % L)
    return vecinos


def generar_pasos_nfold(L: int, J: float, H: float, kBT: float):
    """Generador que emite (red_2d, t_mcs, salto_m, magnetizacion) en cada giro."""
    N = L * L
    p = calcular_probabilidades_glauber(J, H, kBT)
    vecinos = construir_vecinos(L)

    nncls = np.zeros(10, dtype=np.int32)
    nncls[0] = N
    loc = np.zeros((10, N), dtype=np.int32)
    loc[0, :] = np.arange(N, dtype=np.int32)
    looc = np.zeros(N, dtype=np.int32)
    look = np.arange(N, dtype=np.int32)

    espines = np.ones(N, dtype=np.int8)
    M = N
    t = 0

    def mover_spin(spin_i: int, nueva_clase: int):
        clase_vieja = looc[spin_i]
        pos_vieja = look[spin_i]

        ultima_pos = nncls[clase_vieja] - 1
        ultimo_spin = loc[clase_vieja, ultima_pos]
        loc[clase_vieja, pos_vieja] = ultimo_spin
        look[ultimo_spin] = pos_vieja
        nncls[clase_vieja] -= 1

        nueva_pos = nncls[nueva_clase]
        loc[nueva_clase, nueva_pos] = spin_i
        looc[spin_i] = nueva_clase
        look[spin_i] = nueva_pos
        nncls[nueva_clase] += 1

    # Fotograma inicial (t = 0 mcs)
    yield espines.reshape(L, L).copy(), t, 0, M

    while M > 0:
        r_tilde = random.random()
        r_bar = 1.0 - random.random()

        Q = np.cumsum(nncls * p) / N
        Q10 = Q[9]

        m = math.floor(math.log(r_bar) / math.log1p(-Q10)) + 1

        objetivo = r_tilde * Q10
        for k, q_acum in enumerate(Q):
            if objetivo < q_acum:
                break

        pos_aleatoria = random.randrange(nncls[k])
        spin_elegido = loc[k, pos_aleatoria]

        espines[spin_elegido] *= -1

        if k < 5:
            mover_spin(spin_elegido, k + 5)
            for nb in vecinos[spin_elegido]:
                mover_spin(nb, looc[nb] + 1)
            M -= 2
        else:
            mover_spin(spin_elegido, k - 5)
            for nb in vecinos[spin_elegido]:
                mover_spin(nb, looc[nb] - 1)
            M += 2

        t += m
        yield espines.reshape(L, L).copy(), t, m, M


def visualizar_simulacion(L: int = 30, J: float = 1.0, H: float = -1.6, kBT: float = 0.8, intervalo_ms: int = 30):
    """Genera y muestra el reproductor interactivo de la red en Jupyter."""
    N = L * L
    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    red_inicial = np.ones((L, L), dtype=np.int8)
    im = ax.imshow(red_inicial, cmap="coolwarm_r", vmin=-1, vmax=1, interpolation="nearest")

    cbar = plt.colorbar(im, ax=ax, ticks=[-1, 1], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["Abajo (-1)", "Arriba (+1)"])

    ax.set_xticks([])
    ax.set_yticks([])
    titulo = ax.set_title("Tiempo: 0 mcs (0.00 MCSS) | M/N = 1.00")

    def actualizar(fotograma):
        red_2d, t, m, M = fotograma
        im.set_data(red_2d)
        titulo.set_text(
            f"Tiempo: {t:,} mcs ({t / N:.2f} MCSS)\n"
            f"Último salto: +{m:,} mcs | Magnetización M/N: {M / N:.2f}"
        )
        return [im, titulo]

    anim = FuncAnimation(
        fig,
        actualizar,
        frames=generar_pasos_nfold(L, J, H, kBT),
        interval=intervalo_ms,
        blit=False,
        repeat=False,
        save_count=1000,
    )

    plt.tight_layout()
    plt.close(fig)  # Evita que aparezca una imagen fija debajo del vídeo
    display(HTML(anim.to_jshtml()))


# Ejecutar animación
visualizar_simulacion(L=30, J=1.0, H=-1.6, kBT=0.8, intervalo_ms=30)